# Part 1 — Corruption Pipeline

Generates **unanswerable** question variants from DocVQA by corrupting original questions, then verifies them with a Vision LLM judge (LLM-as-a-Judge).

Three corruption types:
- **NLP Entity** — replace a named entity / number in the question with a different one
- **Element** — replace a document-element reference (table → figure, chart → graph …)
- **Layout** — replace a spatial reference (top → bottom, left → right …)

In [1]:
CONFIG = {
    # ---- dataset ----
    "dataset_name": "VLR-CVC/DocVQA-2026",  # public DocVQA subset, no auth needed
    "split": "test",
    "num_samples": 300,
    # ---- corruption ----
    "corruption_distribution": {"nlp_entity": 0.4, "element": 0.3, "layout": 0.3},
    # ---- judge model ----
    # Must be DIFFERENT from the models benchmarked in Part 2 to avoid circular evaluation.
    # Qwen2.5-VL is specialized for document understanding → reliable judge.
    "judge_model": "Qwen/Qwen2.5-VL-7B-Instruct",
    # ---- output ----
    "data_dir": "data",
    # ---- misc ----
    "seed": 42,
    "window_size": 1,   # pages per window; increase for multi-page doc support
}

In [ ]:
import os
import json
import re
import random
import subprocess
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from PIL import Image
from datasets import load_dataset

# Raise PIL's decompression-bomb limit — DocVQA contains very high-res scans
Image.MAX_IMAGE_PIXELS = None

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
Path(CONFIG["data_dir"]).mkdir(parents=True, exist_ok=True)

print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  Device count : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}  {props.total_memory / 1e9:.1f} GB")

## Load dataset

In [ ]:
print(f"Loading {CONFIG['dataset_name']} …")
dataset = load_dataset(CONFIG["dataset_name"], split=CONFIG["split"])
n = min(CONFIG["num_samples"], len(dataset))
dataset = dataset.select(range(n))
print(f"Loaded {len(dataset)} samples.")

# Inspect actual structure before assuming column names
sample = dataset[0]
print(f"\nColumns : {dataset.column_names}")
print(f"Types   : { {k: type(v).__name__ for k, v in sample.items()} }")

# Map to canonical names used throughout the notebook
# Adjust the right-hand side if your dataset uses different names
_QUESTION_COL = next(c for c in dataset.column_names if c.lower() in ["question", "query", "questions"])
_ANSWERS_COL  = next(c for c in dataset.column_names if c.lower() in ["answers", "answer"])
_IMAGE_COL    = next(c for c in dataset.column_names if c.lower() in ["image", "img", "preview"])
print(f"\nMapped  : question='{_QUESTION_COL}'  answers='{_ANSWERS_COL}'  image='{_IMAGE_COL}'")

sample_q = sample[_QUESTION_COL]
sample_a = sample[_ANSWERS_COL]
q_display = next(iter(sample_q.values())) if isinstance(sample_q, dict) else sample_q
a_display = next(iter(sample_a.values())) if isinstance(sample_a, dict) else sample_a
print(f"\nExample :")
print(f"  Q : {q_display}")
print(f"  A : {a_display}")
sample[_IMAGE_COL]   # display first document image inline

## Corruption resources

In [ ]:
# -- spaCy for NER-based entity corruption --
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
except (ImportError, OSError):
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
    import spacy
    nlp = spacy.load("en_core_web_sm")
print(f"spaCy {spacy.__version__} ready.")

# Replacement pools by spaCy entity label
ENTITY_POOLS = {
    "DATE":     ["January 2020", "March 2019", "December 2021", "Q3 2022",
                 "FY2023", "2018", "2024", "April 2021", "June 2017"],
    "CARDINAL": ["42", "157", "3200", "12", "88", "1000", "250", "7", "500"],
    "MONEY":    ["$500", "$1,200", "€2,500", "$75,000", "£300", "$10 million"],
    "ORG":      ["Acme Corp", "Global Industries", "TechSolutions Inc.",
                 "National Bureau", "Allied Group", "Pacific Holdings"],
    "PERSON":   ["John Smith", "Maria Garcia", "Robert Johnson", "Emily Chen", "David Miller"],
    "GPE":      ["New York", "London", "Tokyo", "Berlin", "Sydney", "Chicago", "Paris"],
    "PERCENT":  ["15%", "42%", "3.5%", "99%", "0.1%", "67.8%", "28%"],
    "ORDINAL":  ["first", "second", "third", "fourth", "fifth"],
    "LOC":      ["North America", "Eastern Europe", "the Pacific", "Southeast Asia"],
    "TIME":     ["9:00 AM", "3:30 PM", "midnight", "noon"],
}

# Document element substitutions
ELEMENT_REPLACEMENTS = {
    "table":    ["figure", "chart", "graph", "appendix", "footnote"],
    "figure":   ["table", "chart", "diagram", "graph"],
    "chart":    ["table", "figure", "graph", "diagram"],
    "graph":    ["table", "figure", "chart", "diagram"],
    "footnote": ["table", "figure", "appendix", "section"],
    "section":  ["table", "figure", "appendix", "chapter"],
    "appendix": ["table", "figure", "section", "footnote"],
    "diagram":  ["table", "figure", "chart", "graph"],
}

# Layout / spatial reference substitutions
LAYOUT_REPLACEMENTS = {
    "top":       ["bottom", "middle"],
    "bottom":    ["top", "middle"],
    "left":      ["right", "center"],
    "right":     ["left", "center"],
    "upper":     ["lower", "middle"],
    "lower":     ["upper", "middle"],
    "first":     ["last", "second"],
    "last":      ["first", "second"],
    "header":    ["footer", "body"],
    "footer":    ["header", "body"],
    "above":     ["below", "beside"],
    "below":     ["above", "beside"],
    "beginning": ["end", "middle"],
    "end":       ["beginning", "middle"],
}


def pick_replacement(pool, original):
    candidates = [x for x in pool if x.lower() != original.lower()]
    return random.choice(candidates) if candidates else None

Resolved 1 package in 120ms
Installed 1 package in 111ms
 + en-core-web-sm==3.8.0 (from https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl)


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
spaCy 3.8.14 ready.


## Corruption functions

In [ ]:
def corrupt_nlp_entity(question):
    """Replace a spaCy-detected entity or inline number with a different value."""
    doc = nlp(question)
    candidates = [(e.text, e.label_) for e in doc.ents if e.label_ in ENTITY_POOLS]

    if candidates:
        orig_text, label = random.choice(candidates)
        replacement = pick_replacement(ENTITY_POOLS[label], orig_text)
        if replacement:
            corrupted = question.replace(orig_text, replacement, 1)
            if corrupted != question:
                return {"question": corrupted,
                        "detail": {"original": orig_text, "replacement": replacement,
                                   "entity_type": label}}

    # Fallback: replace an inline integer (e.g. "Table 1" → "Table 4")
    m = re.search(r'\b(\d+)\b', question)
    if m:
        orig_num = m.group(1)
        pool = [str(i) for i in range(1, 10) if str(i) != orig_num]
        replacement = random.choice(pool)
        corrupted = question[:m.start()] + replacement + question[m.end():]
        return {"question": corrupted,
                "detail": {"original": orig_num, "replacement": replacement,
                           "entity_type": "CARDINAL"}}
    return None


def corrupt_element(question):
    """Replace a document-element keyword (table, figure, …) with a different one."""
    q_lower = question.lower()
    for element, replacements in ELEMENT_REPLACEMENTS.items():
        if re.search(r'\b' + re.escape(element) + r'\b', q_lower):
            replacement = random.choice(replacements)
            corrupted = re.sub(r'\b' + re.escape(element) + r'\b',
                               replacement, question, count=1, flags=re.IGNORECASE)
            if corrupted.lower() != question.lower():
                return {"question": corrupted,
                        "detail": {"original": element, "replacement": replacement}}
    return None


def corrupt_layout(question):
    """Replace a spatial / layout keyword (top, left, first, …) with a different one."""
    q_lower = question.lower()
    for term, replacements in LAYOUT_REPLACEMENTS.items():
        if re.search(r'\b' + re.escape(term) + r'\b', q_lower):
            replacement = random.choice(replacements)
            corrupted = re.sub(r'\b' + re.escape(term) + r'\b',
                               replacement, question, count=1, flags=re.IGNORECASE)
            if corrupted.lower() != question.lower():
                return {"question": corrupted,
                        "detail": {"original": term, "replacement": replacement}}
    return None


CORRUPTION_FNS = {
    "nlp_entity": corrupt_nlp_entity,
    "element":    corrupt_element,
    "layout":     corrupt_layout,
}
FALLBACK_ORDER = list(CORRUPTION_FNS.keys())


def apply_corruption(question, preferred_type):
    """Try preferred corruption type; fall back to others if it produces no match."""
    result = CORRUPTION_FNS[preferred_type](question)
    if result is not None:
        return result, preferred_type
    for ctype in FALLBACK_ORDER:
        if ctype == preferred_type:
            continue
        result = CORRUPTION_FNS[ctype](question)
        if result is not None:
            return result, ctype
    return None, None

## Generate corrupted samples

In [ ]:
corruption_types = list(CONFIG["corruption_distribution"].keys())
weights = list(CONFIG["corruption_distribution"].values())

corrupted_data = []
skipped = 0

for idx in tqdm(range(len(dataset)), desc="Corrupting"):
    questions_raw = dataset[idx][_QUESTION_COL]
    answers_raw   = dataset[idx][_ANSWERS_COL]

    # Handle dict, list, and scalar question formats
    if isinstance(questions_raw, dict):
        qa_pairs = [(qid, q, answers_raw.get(qid, []) if isinstance(answers_raw, dict) else answers_raw)
                    for qid, q in questions_raw.items()]
    elif isinstance(questions_raw, list):
        qa_pairs = [(None, q, answers_raw) for q in questions_raw]
    else:
        qa_pairs = [(None, questions_raw, answers_raw)]

    for qid, question, answers in qa_pairs:
        if not isinstance(question, str):
            skipped += 1
            continue
        preferred = random.choices(corruption_types, weights=weights, k=1)[0]
        result, used_type = apply_corruption(question, preferred)

        if result is None:
            skipped += 1
            continue

        corrupted_data.append({
            "idx":                idx,
            "qid":                qid,
            "original_question":  question,
            "original_answers":   answers,
            "corrupted_question": result["question"],
            "corruption_type":    used_type,
            "corruption_detail":  result["detail"],
            "judge_verdict":      None,
        })

print(f"Corrupted : {len(corrupted_data)} / {len(dataset)}  (skipped: {skipped})")
print("Type distribution:", dict(Counter(d["corruption_type"] for d in corrupted_data)))

In [ ]:
print("=== One example per corruption type ===\n")
for ctype in corruption_types:
    example = next((d for d in corrupted_data if d["corruption_type"] == ctype), None)
    if example is None:
        print(f"[{ctype}]  — no examples found\n")
        continue
    det = example["corruption_detail"]
    print(f"[{ctype}]")
    print(f"  Original  : {example['original_question']}")
    print(f"  Corrupted : {example['corrupted_question']}")
    print(f"  Change    : '{det['original']}' → '{det['replacement']}'")
    print()

=== One example per corruption type ===

[nlp_entity]
  Original  : What the location address of NSDA?
  Corrupted : What the location address of Pacific Holdings?
  Change    : 'NSDA' → 'Pacific Holdings'

[element]
  Original  : Who was the chief of the scientific evaluation section?
  Corrupted : Who was the chief of the scientific evaluation appendix?
  Change    : 'section' → 'appendix'

[layout]
  Original  : Which is the Fiscal Year End?
  Corrupted : Which is the Fiscal Year beginning?
  Change    : 'end' → 'beginning'



## LLM-as-a-Judge quality verification

Load the Vision LLM on the HPC GPUs and ask it whether each corrupted question is answerable from the document.
Only samples where the model says **UNANSWERABLE** are kept in the final dataset.

In [ ]:
from transformers import AutoProcessor

try:
    from transformers import AutoModelForImageTextToText
    ModelClass = AutoModelForImageTextToText
except ImportError:                          # transformers < 4.49
    from transformers import AutoModelForCausalLM
    ModelClass = AutoModelForCausalLM

print(f"Loading judge model: {CONFIG['judge_model']} …")
processor = AutoProcessor.from_pretrained(CONFIG["judge_model"])
judge_model = ModelClass.from_pretrained(
    CONFIG["judge_model"],
    device_map="auto",          # spreads across all available GPUs automatically
    dtype=torch.bfloat16,
)
judge_model.eval()
print("Ready.")
print("Device map:", judge_model.hf_device_map)

Loading judge model: Qwen/Qwen2.5-VL-7B-Instruct …


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 729/729 [00:05<00:00, 129.62it/s]


Ready.
Device map: {'model.visual': 0, 'model.language_model.embed_tokens': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 1, 'model.language_model.layers.3': 1, 'model.language_model.layers.4': 1, 'model.language_model.layers.5': 1, 'model.language_model.layers.6': 1, 'model.language_model.layers.7': 1, 'model.language_model.layers.8': 1, 'model.language_model.layers.9': 1, 'model.language_model.layers.10': 1, 'model.language_model.layers.11': 1, 'model.language_model.layers.12': 2, 'model.language_model.layers.13': 2, 'model.language_model.layers.14': 2, 'model.language_model.layers.15': 2, 'model.language_model.layers.16': 2, 'model.language_model.layers.17': 2, 'model.language_model.layers.18': 2, 'model.language_model.layers.19': 2, 'model.language_model.layers.20': 2, 'model.language_model.layers.21': 2, 'model.language_model.layers.22': 3, 'model.language_model.layers.23': 3, 'model.language_model.layers.24': 3, 'model

In [ ]:
JUDGE_PROMPT = (
    "You are a quality control system for a document question answering benchmark.\n"
    "Examine the document image carefully.\n"
    "Can the following question be answered solely from the content visible in the document?\n\n"
    "Question: {question}\n\n"
    "Reply with one word only: ANSWERABLE or UNANSWERABLE"
)

_input_device = next(judge_model.parameters()).device
_MAX_SIDE = 2048  # Qwen2.5-VL internal resolution cap; anything larger is downsampled anyway


def _prepare_image(img: Image.Image) -> Image.Image:
    """Convert to RGB and cap the longest side to _MAX_SIDE to avoid OOM on huge scans."""
    img = img.convert("RGB")
    w, h = img.size
    if max(w, h) > _MAX_SIDE:
        scale = _MAX_SIDE / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return img


def judge_unanswerability(pil_image: Image.Image, question: str) -> str:
    pil_image = _prepare_image(pil_image)
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": JUDGE_PROMPT.format(question=question)},
    ]}]
    text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(
        text=text, images=[pil_image], return_tensors="pt"
    ).to(_input_device)

    with torch.no_grad():
        out = judge_model.generate(**inputs, max_new_tokens=10, do_sample=False)

    input_len = inputs["input_ids"].shape[1]
    response = processor.decode(
        out[0][input_len:], skip_special_tokens=True
    ).strip().upper()
    return "UNANSWERABLE" if "UNANSWERABLE" in response else "ANSWERABLE"

In [ ]:
verified_data = []
rejected = 0

for item in tqdm(corrupted_data, desc="Judging"):
    img = dataset[item["idx"]][_IMAGE_COL]
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img).convert("RGB")

    verdict = judge_unanswerability(img, item["corrupted_question"])
    item["judge_verdict"] = verdict

    if verdict == "UNANSWERABLE":
        verified_data.append(item)
    else:
        rejected += 1

print(f"\nPassed (unanswerable) : {len(verified_data)}")
print(f"Rejected (answerable) : {rejected}")
print(f"Pass rate             : {len(verified_data) / max(len(corrupted_data), 1):.1%}")

Judging: 100%|██████████| 120/120 [02:37<00:00,  1.32s/it]


Passed (unanswerable) : 116
Rejected (answerable) : 4
Pass rate             : 96.7%


## Statistics & examples

In [ ]:
df = pd.DataFrame(verified_data)
print(f"=== Final Dataset: {len(df)} samples ===\n")
print("By corruption type:")
print(df["corruption_type"].value_counts().to_string())

print("\n--- Sample corruptions ---")
for ctype in df["corruption_type"].unique():
    row = df[df["corruption_type"] == ctype].iloc[0]
    print(f"\n[{ctype}]")
    print(f"  Original  : {row['original_question']}")
    print(f"  Corrupted : {row['corrupted_question']}")
    print(f"  Change    : {row['corruption_detail']}")
    print(f"  Orig. ans : {row['original_answers']}")

=== Final Dataset: 116 samples ===

By corruption type:
corruption_type
nlp_entity    105
layout          6
element         5

--- Sample corruptions ---

[nlp_entity]
  Original  : What the location address of NSDA?
  Corrupted : What the location address of Pacific Holdings?
  Change    : {'original': 'NSDA', 'replacement': 'Pacific Holdings', 'entity_type': 'ORG'}
  Orig. ans : ['1128 SIXTEENTH ST., N. W., WASHINGTON, D. C. 20036', '1128 sixteenth st., N. W., washington, D. C. 20036']

[element]
  Original  : Who was the chief of the scientific evaluation section?
  Corrupted : Who was the chief of the scientific evaluation appendix?
  Change    : {'original': 'section', 'replacement': 'appendix'}
  Orig. ans : ['Dr. Joseph C. Hwang']

[layout]
  Original  : Which is the Fiscal Year End?
  Corrupted : Which is the Fiscal Year beginning?
  Change    : {'original': 'end', 'replacement': 'beginning'}
  Orig. ans : ['August 31, 1963']


## Save final dataset

In [ ]:
out_dir  = Path(CONFIG["data_dir"])
img_dir  = out_dir / "images"
img_dir.mkdir(exist_ok=True)

for item in tqdm(verified_data, desc="Saving images"):
    img_path = img_dir / f"{item['idx']:04d}.jpg"
    if not img_path.exists():
        img = dataset[item["idx"]][_IMAGE_COL]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img).convert("RGB")
        img.save(img_path, quality=85)
    item["image_path"] = str(img_path)

dataset_path = out_dir / "corrupted_dataset.json"
with open(dataset_path, "w") as f:
    json.dump(verified_data, f, indent=2)

print(f"Saved {len(verified_data)} samples → {dataset_path}")
print(f"Images                → {img_dir}")

Saving images: 100%|██████████| 116/116 [00:01<00:00, 60.07it/s]

Saved 116 samples → data/corrupted_dataset.json
Images                → data/images
